Key Deliverables 
- EntityExtractor class with regex patterns for numeric entities ✅
- Amenity detection using Week 1 taxonomy ✅
- Labeled dataset: 200-300 remarks with entity spans ✅
- Evaluation script calculating precision/recall/F1 
- Target: 85%+ F1 score on test set 
- Error analysis identifying top failure patterns 

Objective: Build a program that reads a real estate listing description/remarks and pull out important information into a structured format

In [195]:
import re 
from collections import Counter
import json
import ollama
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score
from io import StringIO
import sys
import math
sys.path.append("../scripts")

In [196]:
import importlib
import w2_text_cleaning

importlib.reload(w2_text_cleaning)

from w2_text_cleaning import TextCleaner

========== REMARKS PROFILE ==========

Total listings: 1000
Null rate: 0.00%
Average length: 1290.09 characters

Price mentions: 57
Measurement mentions: 261
Room dimensions: 7

HTML tags: 0
Unicode usages: 394
Smart quotes: 276
Whitespace issues: 482
APN Counts: 3
Phone Number Counts: 0

Known abbreviations
ft       251
sq       237
condo    139
hoa      134
adu      122
bbq      104
rv       70
ev       58
hvac     57
ac       45

Unknown abbreviations
altos    3
sony     3
ages     3
wings    3
pulls    3
hemet    3
harte    3
rents    3
sj       3
tools    3
finds    3
items    3
lofts    3
gpm      2
draws    2
verde    2
lies     2
rises    2
tells    2
baja     2

Top 20 words
home         2383
living       1964
room         1233
offers       1073
space        1068
kitchen      953
private      849
bedroom      832
s            824
dining       815
2            796
features     761
you          750
area         690
located      668
spacious     667
bedrooms     666
new          

In [197]:
with open("../data/processed/taxonomy.json", "r") as f:
    taxonomy = json.load(f)
print("Total terms:", len(taxonomy["terms"]))
print(taxonomy["categories"])
print()
term_counts = Counter(term["category"] for term in taxonomy["terms"])

print("Term Counts per Category")
for category in taxonomy["categories"]:
    count = term_counts.get(category, 0)
    print(f"{category}: {count} terms")

Total terms: 200
['kitchen', 'flooring', 'outdoor', 'parking', 'housing_layout', 'housing_features', 'location', 'community_amenities']

Term Counts per Category
kitchen: 22 terms
flooring: 13 terms
outdoor: 31 terms
parking: 20 terms
housing_layout: 43 terms
housing_features: 41 terms
location: 28 terms
community_amenities: 2 terms


In [198]:
print(taxonomy['terms'])

[{'id': 1, 'term': 'primary suite', 'category': 'housing_layout', 'frequency': 336}, {'id': 2, 'term': 'natural light', 'category': 'housing_features', 'frequency': 294}, {'id': 3, 'term': 'living room', 'category': 'housing_layout', 'frequency': 283}, {'id': 4, 'term': 'living space', 'category': 'outdoor', 'frequency': 263}, {'id': 5, 'term': 'floor plan', 'category': 'flooring', 'frequency': 261}, {'id': 6, 'term': 'stainless steel', 'category': 'kitchen', 'frequency': 193}, {'id': 7, 'term': 'everyday living', 'category': 'parking', 'frequency': 186}, {'id': 8, 'term': 'stainless steel appliances', 'category': 'kitchen', 'frequency': 172}, {'id': 9, 'term': 'shopping dining', 'category': 'location', 'frequency': 171}, {'id': 10, 'term': 'located near', 'category': 'location', 'frequency': 159}, {'id': 11, 'term': 'conveniently located', 'category': 'location', 'frequency': 158}, {'id': 12, 'term': 'home office', 'category': 'housing_layout', 'frequency': 143}, {'id': 13, 'term': 'p

In [199]:
class EntityExtractor: 
    def __init__(self):
        with open("../data/processed/taxonomy.json", "r") as f:
            taxonomy = json.load(f)
        self.taxonomy = taxonomy['terms']
    def extract_bedrooms(self, text): 
        patterns = [
        # Numeric values
        r'\b(\d+)\s*[- ]?(?:bed|beds|bedroom|bedrooms|br|bd|bdrm|bdrms)\b', 
        # Number words
        r'(one|two|three|four|five|six|seven|eight|nine|ten)[- ]?(?:bed|beds|bedroom|bedrooms|br|bd|bdrm|bdrms)\b'
        ]
        if re.search(r"\bone of (the )?bedrooms\b", text, re.IGNORECASE):
            return ">1"
        number_words = {
        "one": 1,
        "two": 2,
        "three": 3,
        "four": 4,
        "five": 5,
        "six": 6,
        "seven": 7,
        "eight": 8,
        "nine": 9,
        "ten": 10
        }
        # Exact bedroom counts
        for pattern in patterns: 
            match = re.search(pattern, text, re.I) 
            if match: 
                value = match.group(1).lower()
                if value.isdigit():
                    return int(value)
                return number_words[value]
            
        # Ordinal bedroom references
        ordinal_map = {
        "primary": 1,
        "first": 1,
        "second": 2,
        "third": 3,
        "fourth": 4,
        "fifth": 5,
        "sixth": 6,
        "seventh": 7,
        "eighth": 8,
        "ninth": 9,
        "tenth": 10
        }
        matches = re.findall(
            r'\b(primary|first|second|third|fourth|fifth|sixth|seventh|eighth|ninth|tenth)\s+(?:bedroom|bed)\b',
            text,
            re.I
        )
        if matches:
            return max(ordinal_map[word.lower()] for word in matches)
        
        # Plural bedroom mention
        if re.search(r'\b(beds|bedrooms)\b', text, re.I):
            return ">1"
        
        # Singular bedroom mention
        if re.search(r'\b(bed|bedroom)\b', text, re.I):
            return 1
        return None 
  
    def extract_price(self, text):
        if not isinstance(text, str):
            return None
        patterns = [
            # Price explicitly mentioned with $
            r'\$\s?(\d{5,})',
            # Price mentioned with keywords
            r'(?:listed|listing|asking|offered|priced|price|sale)\s*(?:at|for|:)?\s*(\d{5,})'
        ]

        for pattern in patterns:
            match = re.search(
                pattern,
                text,
                re.I
            )
            if match:
                return int(match.group(1))
        return None
    def extract_bathrooms(self, text):
        if not isinstance(text, str):
            return None
        # Normalize Unicode fractions
        text = (
            text.replace("½", ".5")
                .replace("¼", ".25")
                .replace("¾", ".75")
        )
        def normalize_bath(value):
            """
            Normalize bathroom values to match MLS ground truth.
            Half baths are counted as whole bathrooms.
            Example:
                2.5 -> 3
                1.5 -> 2
            """
            value = float(value)
            return int(math.ceil(value))
        patterns = [
            # Numeric bathrooms: 2 bath, 2 baths, 2 bathroom, 2.5 ba
            r'(\d+(?:\.\d+)?)\s*[- ]?(?:bath|baths|bathroom|bathrooms|ba)\b',

            # Written numbers: one bath, two bathrooms
            r'\b(one|two|three|four|five)\s+(?:full\s+|half\s+)?(?:bath|baths|bathroom|bathrooms)\b'
        ]
        number_words = {
            "one": 1,
            "two": 2,
            "three": 3,
            "four": 4,
            "five": 5
        }
        for pattern in patterns:
            match = re.search(pattern, text, re.I)
            if match:
                value = match.group(1).lower()
                # Numeric values
                if value.replace(".", "", 1).isdigit():
                    return normalize_bath(value)
                # Written numbers
                return number_words[value]
        # Half bath mentions without a number
        # Example: "includes a half bath"
        if re.search(r'\bhalf[- ]bath\b', text, re.I):
            return 1
        # Full bath mentions without a number
        # Example: "one full bath"
        if re.search(r'\bfull[- ]bath\b', text, re.I):
            return 1
        # Plural bathroom mention without count
        if re.search(r'\b(baths|bathrooms)\b', text, re.I):
            return ">1"
        # Singular bathroom mention
        if re.search(r'\b(bath|bathroom)\b', text, re.I):
            return 1
        return None
    def extract_sqft(self, text): 
        if not isinstance(text, str):
            return None
        patterns = [
            r'([\d,]+)\s*(?:square\s*feet)\b',
            r'([\d,]+)\s*(?:sq\.?\s*ft\.?)\b',
            r'([\d,]+)\s*sqft\b',
            r'([\d,]+)\s*sf\b'
        ]
        for pattern in patterns:
            match = re.search(pattern, text, re.I)
            if match:
                return int(
                    match.group(1).replace(",", "")
                )
        return None
    def extract_amenities(self, text):
        amenities = []
        text = text.lower()
        for item in self.taxonomy:
            term = item["term"].lower()
            if term in text:
                amenities.append({
                    "term": item["term"],
                    "category": item["category"]
                })
        return amenities
    
    def extract_all(self, text): 
        return { 
        'bedrooms': self.extract_bedrooms(text), 
        'bathrooms': self.extract_bathrooms(text), 
        'price': self.extract_price(text), 
        'sqft': self.extract_sqft(text), 
        'amenities': self.extract_amenities(text) 
        } 


Test cases

In [200]:
extractor = EntityExtractor()

In [201]:
remark = '''three full bathroom'''
print(extractor.extract_bathrooms(remark))

3


In [202]:
remark = '''2½-bath'''
print(extractor.extract_bathrooms(remark))

3


In [203]:
remark = '''2.5 baths'''
print(extractor.extract_bathrooms(remark))

3


In [204]:
remark = '''two-bedroom'''
print(extractor.extract_bedrooms(remark))

2


In [205]:
remark1 = """
Beautiful home listed at $399000 with updated kitchen.
"""

print(extractor.extract_price(remark1))

399000


In [206]:
# No housing price
remark2 = """
Beautiful estate featuring 4146 square feet.
HOA fee $500 monthly.
"""
print(extractor.extract_price(remark2))

None


In [207]:
remark_ex = '''
This condominium is set up as a great roommate split. Two primary bedroom suites, each on its own floor...
'''
print(extractor.extract_bedrooms(remark_ex))

1


In [208]:
remark_ex2 = '''
Welcome to La Playa Court. Built in 2008, this sophisticated building offers upscale elegance and resort-style amenities just minutes from the beach, local restaurants, and shops. This elegant, rare corner unit features a spacious open floor plan designed for both entertaining and privacy. Upon entering the unit, you come upon the dining area with large window and shutters for privacy. The gourmet chef's kitchen is cleverly set off the dining area and equipped with high-end appliances, granite countertops, travertine stone backsplash and flooring. There is even a window above the sink and a breakfast bar. The open living area is centered by a modern and elegant gas fireplace and access to a private balcony. The primary suite includes a custom walk-in closet and an ensuite spa bath with dual sinks, a soaking tub, and a separate shower. The second bedroom also features a large wall-to-wall closet, with a second full bathroom and linen closet located nearby,
'''
print(extractor.extract_bedrooms(remark_ex2))

2


In [209]:
remark_ex3 = '''
Welcome to this stunning, move-in-ready residence showcasing modern curb appeal and thoughtfully updated interiors throughout. The striking black-and-white exterior creates a sophisticated first impression, complemented by a spacious front yard, custom walkway accents, and an oversized two-car garage. Inside, you'll find an open and airy floor plan featuring vaulted ceilings, contemporary gray wood-look flooring, fresh paint, and abundant natural light. The spacious living area is highlighted by a stylish painted brick fireplace and large sliding glass doors that seamlessly connect indoor and outdoor living spaces. The remodeled kitchen and bathrooms feature elegant finishes, including modern fixtures, upgraded cabinetry, quartz-style countertops, and a sleek vessel sink in the guest bath. Generously sized bedrooms and updated living spaces provide comfort and functionality for today's lifestyle. The private backyard offers endless possibilities for entertaining, relaxation, or future customization.
'''
print(extractor.extract_bedrooms(remark_ex3))

>1


In [210]:
remark_ex4 = ''' 
*AMAZING LOG-STYLE RETREAT IN EAGLE MOUNTAIN ESTATES!** This beautifully maintained single-story home offers level entry, 3 spacious en suite bedrooms (3 master suites), 3 baths, and approximately 2,200 square feet. of comfortable mountain living. The open great room features soaring cathedral ceilings, abundant natural light, and a stunning custom stone fireplace. Situated on a large, partially fenced **10,000 square feet. lot**, the property offers plenty of outdoor space, including a covered front porch and a large refinished rear deck with inviting views. Additional features include an **oversized 2-car attached garage**, **Electric vehicle parking**, and a **tankless water heater**. Recent upgrades include a new Aspen spa, A/C system, whole-house humidifier, garage door, and solar system (**solar negotiable**). An unfinished area beneath the home provides excellent additional storage potential. Ideally located near Big Bear Lake, ski resorts, hiking trails, shopping, and dining. Perfect as a full-time residence, vacation getaway, or short-term rental opportunity. **Furnishings negotiable.** A wonderful mountain escape ready to create lasting memories with family and friends.
'''
print(extractor.extract_bedrooms(remark_ex4))

>1


In [211]:
remark_ex5 = '''Featuring 3 spacious bedrooms'''
print(extractor.extract_bedrooms(remark_ex5))

>1


In [212]:
remark_ex6 = '''The spacious bedroom and well-kept living room'''
print(extractor.extract_bedrooms(remark_ex6))

1


In [213]:
remark_ex = '''Half bath'''
print(extractor.extract_bathrooms(remark_ex))

1


In [214]:
remark_ex = '''Full bath'''
print(extractor.extract_bathrooms(remark_ex))

1


In [215]:
remark_ex = '''2.5 bath'''
print(extractor.extract_bathrooms(remark_ex))

3


In [216]:
df_cleaned = pd.read_csv("../data/processed/cleaned_listing_sample.csv")
df_cleaned.head()

,L_ListingID,L_Address,L_City,beds,baths,price,remarks,cleaned_remarks
0,1169503734,133 Crystal Street,Taft,3.0,2.0,235000,"This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and\r\r\ninvestors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom.\r\r\nIdeal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which\r\r\nmakes it easier to occupy or rent it out!","This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and investors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom. Ideal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which makes it easier to occupy or rent it out!"
1,1154220129,15664 Kadota,Victorville,4.0,4.0,459000,"Beautiful Two-Story Home in Victorville – Spacious, Modern, and Move-In Ready Welcome to this stunning two-story residence offering the perfect blend of comfort, style, and functionality. Located in a desirable Victorville neighborhood, this 4-bedroom, 3.5-bathroom home provides generous living space ideal for families, entertainers, or anyone seeking room to grow. Step inside to an inviting open layout featuring abundant natural light, a spacious living area, and new flooring throughout, as well as a well-appointed kitchen with ample cabinetry and counter space. The main floor includes one bedroom with a full bath attached and a half bathroom convenient for guests and easy access to the backyard, perfect for outdoor gatherings. Upstairs, you’ll find two well-sized bedrooms and a luxurious primary suite complete with a private full bathroom and spacious walk-in closet and a full bathroom to ensure comfort and convenience for family or guests. you will also find a spacious loft for relaxation. The home also offers a two-car garage, a low-maintenance yard, and proximity to shopping, dining, schools, and major commuter routes. Whether you're looking for space, comfort, or modern living, this Victorville gem checks every box. Don’t miss the opportunity to make it yours.","Beautiful Two-Story Home in Victorville - Spacious, Modern, and Move-In Ready Welcome to this stunning two-story residence offering the perfect blend of comfort, style, and functionality. Located in a desirable Victorville neighborhood, this 4-bedroom, 3.5-bathroom home provides generous living space ideal for families, entertainers, or anyone seeking room to grow. Step inside to an inviting open layout featuring abundant natural light, a spacious living area, and new flooring throughout, as well as a well-appointed kitchen with ample cabinetry and counter space. The main floor includes one bedroom with a full bath attached and a half bathroom convenient for guests and easy access to the backyard, perfect for outdoor gatherings. Upstairs, you'll find two well-sized bedrooms and a luxurious primary suite complete with a private full bathroom and spacious walk-in closet and a full bathroom to ensure comfort and convenience for family or guests. you will also find a spacious loft for relaxation. The home also offers a two-car garage, a low-maintenance yard, and proximity to shopping, dining, schools, and major commuter routes. Whether you're looking for space, comfort, or modern living, this Victorville gem checks every box. Don't miss the opportunity to make it yours."
2,1159921951,949 S Manhattan Place 203,Los Angeles,3.0,2.0,689000,"Presenting this exquisite second-floor corner unit nestled in the heart of Koreatown—a remarkable 3-bedroom, 2-bathroom, plus den (enclosed balcony) residence offering 1,125 square feet of sophisticated living space with the rare advantage of no neighbors below. Impeccably turnkey and move-in ready, this stylish home showcases soaring ceilings, elega

In [217]:
df_cleaned[['remarks']].head(1)

,remarks
0,"This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and\r\r\ninvestors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom.\r\r\nIdeal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which\r\r\nmakes it easier to occupy or rent it out!"


Checking extracting amenities

In [218]:
remark = """
Beautiful 4 bedroom home with granite countertops,
luxury vinyl plank flooring, pool, and RV parking.
"""

In [219]:
print(extractor.extract_amenities(remark))

[{'term': 'granite countertops', 'category': 'kitchen'}, {'term': 'plank flooring', 'category': 'flooring'}, {'term': 'vinyl plank flooring', 'category': 'flooring'}, {'term': 'rv parking', 'category': 'parking'}]


Extracting all

In [220]:
df_cleaned.columns

Index(['L_ListingID', 'L_Address', 'L_City', 'beds', 'baths', 'price',
       'remarks', 'cleaned_remarks'],
      dtype='str')

In [221]:
sample_remark = df_cleaned['cleaned_remarks'].iloc[0]

In [222]:
sample_remark

'This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and investors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom. Ideal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which makes it easier to occupy or rent it out!'

In [223]:
print(extractor.extract_all(sample_remark))

{'bedrooms': 2, 'bathrooms': 1, 'price': None, 'sqft': None, 'amenities': []}


With this sample, only the first mentions of the bedrooms and bathrooms were extracted. Looking at the remark, they expanded on what the rear unit consists of. 

In [224]:
example_remark = '''
There are 2.5 beds and 3 bathrooms.
'''

In [225]:
print(extractor.extract_all(sample_remark))

{'bedrooms': 2, 'bathrooms': 1, 'price': None, 'sqft': None, 'amenities': []}


#### Labeled dataset: 200-300 remarks with entity spans 

In [226]:
labeled_sample = df_cleaned.sample(n=400, random_state=422)

In [227]:
#labeled_sample.to_csv('../data/labeled_sample.csv', index=False)
print(len(labeled_sample))

400


In [228]:
labeled_sample_cleaned = labeled_sample.drop(columns=['L_Address', 'remarks', 'L_City', 'L_ListingID'])

In [229]:
labeled_sample_cleaned.head()

,beds,baths,price,cleaned_remarks
606,2.0,1.0,399977,"This inviting 2-bedroom, 1-bath residence sits on a generous 1/3-acre lot in the desirable north downtown area of Yucca Valley, offering a blend of comfort, functionality, and seclusion. The property is framed by attractive landscaping and scenic mountain views, with privacy fencing enclosing both the front and rear yards. A gated outdoor parking area adds convenience and security, while covered patios at the front and back create perfect spots to unwind and enjoy the surroundings. Inside, the home features a bright and open living space highlighted by vaulted, beamed ceilings, tile flooring throughout, and updated windows that bring in plenty of natural light. The kitchen is well-equipped and includes a casual dining nook with sliding glass doors leading to the backyard, along with a separate dining area ideal for entertaining or flexible use. The spacious bathroom includes a tub and shower combination. One bedroom offers built-in storage and comfortable proportions, while the second bedroom is notably large, filled with natural light, and includes a generous closet. An additional bonus room provides extra space suitable for a home office, creative studio, or guest accommodations. This home is ready for immediate enjoyment. With its well-designed layout, recent improvements, and inviting outdoor spaces, this property presents a fantastic opportunity in a sought-after Yucca Valley location. Come tour this property today!"
255,2.0,3.0,389000,"Multi-level townhouse looking for a little love. This condominium is set up as a great roommate split. Two primary bedroom suites, each on its own floor. Main floor has full sized kitchen, living/dining room with fireplace and deck and a half bathroom. Two car garage. Private fenced backyard area. Close to downtown Redlands and an easy commute to Loma Linda. Redlands school district. Homeowners Association includes pool and spa. Unit #D."
258,4.0,4.0,2625000,"Experience refined Central Coast living in this stunning 4,146 square feet. Spanish Colonial estate, perfectly positioned on a private 3 acre parcel within the exclusive gates of Santa Ysabel Ranch. Enjoy sweeping 180° views of vineyards and mountains from nearly every room. This 4-bedroom, 3.5-bath home welcomes you through a grand rotunda entry with soaring ceilings, custom lighting, and travertine flooring throughout. The formal living and dining rooms feature tall Anderson windows framing breathtaking vistas and a cozy gas fireplace. The gourmet kitchen is equipped with GE Monogram stainless appliances, including a six-burner range, double ovens, wine fridge, and granite countertops with maple cabinetry. The adjoining family room and casual dining area create an inviting space for entertaining with seamless indoor-outdoor flow. The spacious primary suite offers privacy, dual walk-in closets, and a luxurious bath with soaking tub and large shower. Additional bedrooms and a flexible office space capture beautiful hillside and sunset views. Outdoor living shines with expansive flagstone patios, multiple sitting areas, and lush landscaping surrounded by heritage oaks - perfect for dining, relaxing, or entertaining under the stars. Additional features include an epoxy-finished 3-car garage with custom cabinetry, Culligan water system, American district telegraph security, and custom lighting throughout. Santa Ysabel Ranch offers 24/7 gated security, walking trails, tennis courts, and a serene lake setting - an unparalleled blend of luxury, privacy, and natural beauty."
301,3.0,4.0,3325000,"Wine country estate set across 52 acres in the heart of the Paso Robles AVA, commanding unobstructed views over 28.5 planted acres of Cabernet Sauvignon and expansive sunset horizons. The 3,869 square feet. custom residence is thoughtfully positioned to capture the surrounding vineyard vistas from nearly every room. A dramatic entry introduces Travertine flooring, exposed beams, and walls of windo

sqft (ground truth)

In [230]:
labeled_sample_cleaned['sqft'] = None

In [231]:
sqft_rows = labeled_sample_cleaned[
    labeled_sample_cleaned["cleaned_remarks"].str.contains(
        r'\d[\d,]*(?:[±+]|\+/-)?\s*(?:square\s+feet|sq\s*ft|sqft|\bsf\b)',
        case=False,
        na=False,
        regex=True
    )
]
len(sqft_rows)

139

In [232]:
def sqft_context(text, window=35):
    if pd.isna(text):
        return None

    match = re.search(
        r'\d[\d,]*(?:[±+]|\+/-)?\s*(?:square\s+feet|sq\s*ft|sqft|\bsf\b)',
        text,
        flags=re.IGNORECASE
    )

    if not match:
        return None

    start = max(0, match.start() - window)
    end = min(len(text), match.end() + window)

    return text[start:end]

In [233]:
labeled_sample_cleaned["sqft_context"] = labeled_sample_cleaned["cleaned_remarks"].apply(sqft_context)

In [234]:
labeled_sample_cleaned[labeled_sample_cleaned["sqft_context"].notna()][["sqft_context"]]

,sqft_context
258,"tral Coast living in this stunning 4,146 square feet. Spanish Colonial estate, perfectl"
301,"and expansive sunset horizons. The 3,869 square feet. custom residence is thoughtfully"
944,"nal versatility. Inside, more than 1,500 square feet of light filled living space showc"
288,"ring 5 bedrooms, 4.5 bathrooms and 3,606 square feet with a bright open layout and SPAC"
229,"nt single-family home spans nearly 3,300 square feet. and offers a perfect blend of com"
...,...
922,"N END UNIT, THIS Condominium BOAST 2167 square feet WITH MANY COMMUNITY AMNETIES, INCL"
275,"5-bedroom, 3.5-bath home offering 3,012 square feet. of living space on a lush and pri"
492,"full bathrooms, and approximately 1,318 square feet of single-level living space, with"
848,e ranging from 2 to 4 bedrooms and 1435 square feet to 2339 square feet. Stop on by an


In [235]:
def sqft_phrase(text):
    if pd.isna(text):
        return None

    match = re.search(
        r'\d[\d,]*(?:[±+]|\+/-)?\s*(?:square\s+feet|sq\s*ft|sqft|\bsf\b)',
        text,
        flags=re.IGNORECASE
    )

    return match.group(0) if match else None

In [236]:
labeled_sample_cleaned["sqft_phrase"] = labeled_sample_cleaned["cleaned_remarks"].apply(sqft_phrase)

In [237]:
labeled_sample_cleaned['sqft_phrase'].notna().sum()

np.int64(139)

In [238]:
labeled_sample_cleaned["sqft_phrase"] = labeled_sample_cleaned["cleaned_remarks"].apply(sqft_phrase)
labeled_sample_cleaned["sqft_context"] = labeled_sample_cleaned["cleaned_remarks"].apply(sqft_context)
labeled_sample_cleaned["sqft"] = None

In [239]:
df_cleaned['cleaned_remarks'].loc[258]

'Experience refined Central Coast living in this stunning 4,146 square feet. Spanish Colonial estate, perfectly positioned on a private 3 acre parcel within the exclusive gates of Santa Ysabel Ranch. Enjoy sweeping 180° views of vineyards and mountains from nearly every room. This 4-bedroom, 3.5-bath home welcomes you through a grand rotunda entry with soaring ceilings, custom lighting, and travertine flooring throughout. The formal living and dining rooms feature tall Anderson windows framing breathtaking vistas and a cozy gas fireplace. The gourmet kitchen is equipped with GE Monogram stainless appliances, including a six-burner range, double ovens, wine fridge, and granite countertops with maple cabinetry. The adjoining family room and casual dining area create an inviting space for entertaining with seamless indoor-outdoor flow. The spacious primary suite offers privacy, dual walk-in closets, and a luxurious bath with soaking tub and large shower. Additional bedrooms and a flexible

In [240]:
def label_sqft(phrase):
    if pd.isna(phrase):
        return None

    match = re.search(r'(\d[\d,]*)', phrase)

    if match:
        return int(match.group(1).replace(",", ""))

    return None

In [241]:
labeled_sample_cleaned["sqft"] = labeled_sample_cleaned["sqft_phrase"].apply(label_sqft)

In [242]:
labeled_sample_cleaned['sqft'].notna().sum()

np.int64(139)

In [243]:
labeled_sample_cleaned.columns

Index(['beds', 'baths', 'price', 'cleaned_remarks', 'sqft', 'sqft_context',
       'sqft_phrase'],
      dtype='str')

In [244]:
labeled_sample_cleaned['sqft_context'].notna().sum()

np.int64(139)

In [245]:
labeled_sample_cleaned['sqft_phrase'].notna().sum()

np.int64(139)

In [246]:
labeled_cleaned = labeled_sample_cleaned.drop(columns=['sqft_phrase', 'sqft_context'])

In [247]:
labeled_cleaned["sqft"] = labeled_cleaned["sqft"].fillna("None")

In [248]:
labeled_cleaned['sqft'].isna().sum()

np.int64(0)

Amendities (ground truth)

In [249]:
# Using taxonomy as a helper/resource
def suggest_amenities(text):
    suggestions = []

    for term in taxonomy_terms:
        if term.lower() in text.lower():
            suggestions.append(term)

    return suggestions

taxonomy_terms = sorted(
    [term["term"] for term in taxonomy["terms"]],
    key=len,
    reverse=True
)

In [250]:
labeled_cleaned["candidate_amenities"] = (
    labeled_cleaned["cleaned_remarks"]
    .apply(suggest_amenities)
)

In [251]:
labeled_cleaned['candidate_amenities']

606                                                                                                                              [flooring throughout, sliding glass doors, mountain views, second bedroom, natural light, tile flooring, covered patio, outdoor space, sliding glass, living space, home office, glass doors]
255                                                                                                                                                                                                                                                                                 [primary bedroom, dining room, car garage]
258                                                               [spacious primary suite, primary suite offers, granite countertops, flooring throughout, additional bedrooms, spacious primary, gourmet kitchen, primary suite, tennis courts, gas fireplace, suite offers, family room, dining room, car garage, bath home]
301                                        

Go through the suggested amenities

In [252]:
#labeled_cleaned.to_csv("../data/labeled_cleaned.csv")

In [253]:
labeled_cleaned.columns

Index(['beds', 'baths', 'price', 'cleaned_remarks', 'sqft',
       'candidate_amenities'],
      dtype='str')

In [254]:
labeled_cleaned[
    ['sqft', 'cleaned_remarks']
]

,sqft,cleaned_remarks
606,None,"This inviting 2-bedroom, 1-bath residence sits on a generous 1/3-acre lot in the desirable north downtown area of Yucca Valley, offering a blend of comfort, functionality, and seclusion. The property is framed by attractive landscaping and scenic mountain views, with privacy fencing enclosing both the front and rear yards. A gated outdoor parking area adds convenience and security, while covered patios at the front and back create perfect spots to unwind and enjoy the surroundings. Inside, the home features a bright and open living space highlighted by vaulted, beamed ceilings, tile flooring throughout, and updated windows that bring in plenty of natural light. The kitchen is well-equipped and includes a casual dining nook with sliding glass doors leading to the backyard, along with a separate dining area ideal for entertaining or flexible use. The spacious bathroom includes a tub and shower combination. One bedroom offers built-in storage and comfortable proportions, while the second bedroom is notably large, filled with natural light, and includes a generous closet. An additional bonus room provides extra space suitable for a home office, creative studio, or guest accommodations. This home is ready for immediate enjoyment. With its well-designed layout, recent improvements, and inviting outdoor spaces, this property presents a fantastic opportunity in a sought-after Yucca Valley location. Come tour this property today!"
255,None,"Multi-level townhouse looking for a little love. This condominium is set up as a great roommate split. Two primary bedroom suites, each on its own floor. Main floor has full sized kitchen, living/dining room with fireplace and deck and a half bathroom. Two car garage. Private fenced backyard area. Close to downtown Redlands and an easy commute to Loma Linda. Redlands school district. Homeowners Association includes pool and spa. Unit #D."
258,4146.0,"Experience refined Central Coast living in this stunning 4,146 square feet. Spanish Colonial estate, perfectly positioned on a private 3 acre parcel within the exclusive gates of Santa Ysabel Ranch. Enjoy sweeping 180° views of vineyards and mountains from nearly every room. This 4-bedroom, 3.5-bath home welcomes you through a grand rotunda entry with soaring ceilings, custom lighting, and travertine flooring throughout. The formal living and dining rooms feature tall Anderson windows framing breathtaking vistas and a cozy gas fireplace. The gourmet kitchen is equipped with GE Monogram stainless appliances, including a six-burner range, double ovens, wine fridge, and granite countertops with maple cabinetry. The adjoining family room and casual dining area create an inviting space for entertaining with seamless indoor-outdoor flow. The spacious primary suite offers privacy, dual walk-in closets, and a luxurious bath with soaking tub and large shower. Additional bedrooms and a flexible office space capture beautiful hillside and sunset views. Outdoor living shines with expansive flagstone patios, multiple sitting areas, and lush landscaping surrounded by heritage oaks - perfect for dining, relaxing, or entertaining under the stars. Additional features include an epoxy-finished 3-car garage with custom cabinetry, Culligan water system, American district telegraph security, and custom lighting throughout. Santa Ysabel Ranch offers 24/7 gated security, walking trails, tennis courts, and a serene lake setting - an unparalleled blend of luxury, privacy, and natural beauty."
301,3869.0,"Wine country estate set across 52 acres in the heart of the Paso Robles AVA, commanding unobstructed views over 28.5 planted acres of Cabernet Sauvignon and expansive sunset horizons. The 3,869 square feet. custom residence is thoughtfully positioned to capture the surrounding vineyard vistas from nearly every room. A dramatic entry introduces Travertine flooring, exposed beams, and walls of windows that frame the landscape beyond. The open-conce

Keep in mind: The amentities are meant to represent the features that the property offers, not surrounding words or phrases that happen to contain those features. I will also avoid any marketing phrases. An amenity will represent a tangible property feature, space, service, or location advantage that the property provides. So, if there are any phrases that are not supportive of this, they will be removed (to keep ground truth)

In [255]:
all_amenities = [
    amenity
    for row in labeled_cleaned["candidate_amenities"]
    for amenity in row
]

amenity_counts = Counter(all_amenities)

amenity_counts.most_common()

[('living space', 138),
 ('primary suite', 130),
 ('natural light', 110),
 ('floor plan', 104),
 ('car garage', 98),
 ('living room', 96),
 ('everyday living', 71),
 ('conveniently located', 64),
 ('full bath', 63),
 ('stainless steel', 62),
 ('located near', 61),
 ('easy access', 59),
 ('stainless steel appliances', 56),
 ('home office', 51),
 ('family room', 50),
 ('recessed lighting', 49),
 ('living spaces', 48),
 ('quartz countertops', 45),
 ('full bathroom', 45),
 ('laundry room', 42),
 ('conveniently located near', 41),
 ('dining room', 39),
 ('granite countertops', 38),
 ('abundant natural light', 37),
 ('ideally located', 36),
 ('outdoor space', 35),
 ('suite offers', 35),
 ('bath home', 35),
 ('fully remodeled', 35),
 ('spacious primary', 34),
 ('gated community', 34),
 ('spacious living', 33),
 ('main level', 33),
 ('open floor', 32),
 ('primary bedroom', 31),
 ('primary suite offers', 31),
 ('direct access', 30),
 ('covered patio', 29),
 ('spacious primary suite', 29),
 ('fi

In [256]:
len(amenity_counts)

156

In [257]:
removed_amenities= [
 'everyday living', 'conveniently located', 'located near', 
 'easy access', 'conveniently located near', 'ideally located', 
 'spacious primary', 'flooring throughout', 'fully remodeled', 
 'primary suite offers', 'suite offers', 'direct access', 'flows seamlessly', 
 'convenient access', 'everyday convenience', 'located within', 
 'prime location', 'brand new', 'ample space', 'home located', 
 'beautifully remodeled', 'ideally located near', 'comfortable living space', 
 'enjoy access', 'endless possibilities', 'seamless flow', 'effortless entertaining', 
 'primary suite serves', 'suite serves', 'kitchen showcases', 'everyday comfort', 
 'every detail', 'primary suite features', 'suite features','suite includes', 
 'floor plan features', 'peaceful retreat', 'kitchen featuring', 'level features', 
 'spacious home', 'bright open', 'patio perfect', 'kitchen equipped',
 'features spacious','bedrooms including', 'seamless living', 'located minutes',
 'feet living space', 'space home'
]

In [258]:
labeled_cleaned["candidate_amenities"] = labeled_cleaned["candidate_amenities"].apply(
    lambda amenities: [
        x for x in amenities
        if x not in removed_amenities
    ]
)

In [259]:
all_amenities = [
    amenity
    for row in labeled_cleaned["candidate_amenities"]
    for amenity in row
]

amenity_counts = Counter(all_amenities)

amenity_counts.most_common()

[('living space', 138),
 ('primary suite', 130),
 ('natural light', 110),
 ('floor plan', 104),
 ('car garage', 98),
 ('living room', 96),
 ('full bath', 63),
 ('stainless steel', 62),
 ('stainless steel appliances', 56),
 ('home office', 51),
 ('family room', 50),
 ('recessed lighting', 49),
 ('living spaces', 48),
 ('quartz countertops', 45),
 ('full bathroom', 45),
 ('laundry room', 42),
 ('dining room', 39),
 ('granite countertops', 38),
 ('abundant natural light', 37),
 ('outdoor space', 35),
 ('bath home', 35),
 ('gated community', 34),
 ('spacious living', 33),
 ('main level', 33),
 ('open floor', 32),
 ('primary bedroom', 31),
 ('covered patio', 29),
 ('spacious primary suite', 29),
 ('fitness center', 29),
 ('kitchen features', 29),
 ('updated kitchen', 28),
 ('open floor plan', 28),
 ('parking space', 27),
 ('golf course', 27),
 ('vaulted ceilings', 26),
 ('mountain views', 25),
 ('wood flooring', 25),
 ('private balcony', 25),
 ('hardwood floors', 25),
 ('center island', 25)

In [260]:
len(amenity_counts)

108

In [261]:
labeled_cleaned = labeled_cleaned.rename(columns={'candidate_amenities': 'amenities'})

Using the extractor class

In [262]:
labeled_extract = labeled_cleaned.copy()

In [263]:
labeled_extract.shape

(400, 6)

Extracting bedrooms

In [264]:
labeled_extract['extracted_bd'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_bedrooms
)

In [265]:
labeled_extract['extracted_bd'].isna().sum()

np.int64(61)

In [266]:
missing_bd = labeled_extract[
    labeled_extract["extracted_bd"].isna()
].copy()

In [267]:
missing_bd.shape

(61, 7)

In [268]:
bed_pattern = (
    r"\b("
    r"bed|beds|bedroom|bedrooms|"
    r"br|bd|bdrm|bdrms"
    r")\b"
)

mentioned_bed = missing_bd[
    missing_bd["cleaned_remarks"].str.contains(
        bed_pattern,
        case=False,
        regex=True,
        na=False
    )
]

C:\Users\mayab\AppData\Local\Temp\ipykernel_13260\2472845788.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  missing_bd["cleaned_remarks"].str.contains(


In [269]:
mentioned_bed.shape

(0, 7)

In [270]:
mentioned_bed.columns

Index(['beds', 'baths', 'price', 'cleaned_remarks', 'sqft', 'amenities',
       'extracted_bd'],
      dtype='str')

In [271]:
for i, row in mentioned_bed.iterrows():
    print(f"Row {i}")
    print(f"Ground truth beds: {row['beds']}")
    print(f"Extracted beds: {row['extracted_bd']}")
    print(row["cleaned_remarks"])
    print("-" * 100)

Now that there are no more bedroom mentions, we can assume 53 remarks did not mention anything about bedrooms, so we will remove those rows from our analysis. 

In [272]:
labeled_extract = labeled_extract[~labeled_extract["extracted_bd"].isna()]

In [273]:
labeled_extract.shape

(339, 7)

Extracting bathrooms

In [274]:
labeled_extract['extracted_ba'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_bathrooms
)

In [275]:
labeled_extract['extracted_ba'].isna().sum()

np.int64(34)

In [276]:
missing_br = labeled_extract[
    labeled_extract["extracted_ba"].isna()
].copy()

In [277]:
missing_br.shape

(34, 8)

In [278]:
bath_pattern = (
    r"\b("
    r"bath|baths|bathroom|bathrooms|"
    r"ba|"
    r"half bath|half-bath|"
    r"full bath|full-bath"
    r")\b"
)

mentioned_bath = missing_br[
    missing_br["cleaned_remarks"].str.contains(
        bath_pattern,
        case=False,
        regex=True,
        na=False
    )
]

C:\Users\mayab\AppData\Local\Temp\ipykernel_13260\3455533733.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  missing_br["cleaned_remarks"].str.contains(


In [279]:
mentioned_bath.shape

(0, 8)

In [280]:
for i, row in mentioned_bath.iterrows():
    print(f"Row {i}")
    print(f"Ground truth bathrooms: {row['baths']}")
    print(f"Extracted bathrooms: {row['extracted_ba']}")
    print(row["cleaned_remarks"])
    print("-" * 100)

In [281]:
labeled_extract = labeled_extract[~labeled_extract["extracted_ba"].isna()]

In [282]:
labeled_extract.shape

(305, 8)

Extracting price

In [283]:
'''response = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': 'Say hello in one sentence.'}]
)
print(response['message']['content'])'''

"response = ollama.chat(\n    model='llama3.2',\n    messages=[{'role': 'user', 'content': 'Say hello in one sentence.'}]\n)\nprint(response['message']['content'])"

In [284]:
def generate_price_dataset(
    n_examples=25,
    model="llama3.2"
):

    prompt = f"""
Generate exactly {n_examples} MLS listing remarks.

Return a JSON object with this exact structure:

{{
    "listings": [
        {{
            "remark": "listing description",
            "price": 500000
        }}
    ]
}}

The "listings" array MUST contain exactly {n_examples} objects.

Rules:

- Each remark should be realistic MLS listing text.
- Each remark should be 50-150 words.
- 80% should mention the home's listing price.
- 20% should not mention the price.

Include difficult examples:
- HOA fees
- property taxes
- closing credits
- renovation costs

Example:

"Beautiful home listed at $850,000. HOA dues are $350/month and annual taxes are $7,500."

The correct price is:

850000

If the listing price is not mentioned:

price = null

The price field must ONLY represent the home's listing price.

Return ONLY JSON.
"""

    response = ollama.chat(
        model=model,
        format="json",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    data = json.loads(response.message.content)

    # Extract the listings array
    listings = data["listings"]

    return pd.DataFrame(listings)

In [285]:
'''test = generate_price_dataset(3)

print(test)
print(len(test))'''

'test = generate_price_dataset(3)\n\nprint(test)\nprint(len(test))'

In [286]:
'''all_batches = []

for i in range(8):
    print(f"Generating batch {i+1}/8")
    batch = generate_price_dataset(25)
    all_batches.append(batch)


price_dataset = pd.concat(
    all_batches,
    ignore_index=True
)

price_dataset.shape'''

'all_batches = []\n\nfor i in range(8):\n    print(f"Generating batch {i+1}/8")\n    batch = generate_price_dataset(25)\n    all_batches.append(batch)\n\n\nprice_dataset = pd.concat(\n    all_batches,\n    ignore_index=True\n)\n\nprice_dataset.shape'

In [287]:
'''price_dataset.to_csv(
    "../data/price_dataset.csv",
    index=False
)'''

'price_dataset.to_csv(\n    "../data/price_dataset.csv",\n    index=False\n)'

In [288]:
price_dataset = pd.read_csv("../data/price_dataset.csv")

In [289]:
price_dataset.head()

,remark,price
0,"Beautiful home listed at $1,200,000. The property features 4 spacious bedrooms, 3 full bathrooms, and a large master suite with a soaking tub. HOA fees are $500/month and annual taxes are $10,000.",1200000.0
1,"Stunning lakefront home listed at $900,000. The property boasts 5 bedrooms, 4 full bathrooms, and a private dock perfect for fishing. Property taxes are $8,000/year. Closing credits available up to $50,000.",900000.0
2,"Charming bungalow listed at $600,000. This home features 3 bedrooms, 2 full bathrooms, and a cozy living room with a fireplace. HOA fees are $200/month.",600000.0
3,"Luxury estate listed at $1,800,000. The property offers 6 spacious bedrooms, 5 full bathrooms, and a private movie theater. Annual property taxes are $20,000. Closing credits available up to $100,000.",1800000.0
4,"Cozy home listed at $450,000. This property features 3 bedrooms, 2 full bathrooms, and a large backyard perfect for outdoor entertaining. HOA fees are $300/month.",450000.0


In [290]:
cleaner = TextCleaner()

In [291]:
remark = '''Beautiful home listed at $1,200,000.'''
print(cleaner.clean_text(remark))

Beautiful home listed at $1200000.


In [292]:
price_dataset['cleaned_remark'] = price_dataset['remark'].apply(
    cleaner.clean_text
)

In [293]:
price_dataset['cleaned_remark'].head()

0    Beautiful home listed at $1200000. The property features 4 spacious bedrooms, 3 full bathrooms, and a large master suite with a soaking tub. Homeowners Association fees are $500/month and annual taxes are $10000.
1             Stunning lakefront home listed at $900000. The property boasts 5 bedrooms, 4 full bathrooms, and a private dock perfect for fishing. Property taxes are $8000/year. Closing credits available up to $50000.
2                                              Charming bungalow listed at $600000. This home features 3 bedrooms, 2 full bathrooms, and a cozy living room with a fireplace. Homeowners Association fees are $200/month.
3                    Luxury estate listed at $1800000. The property offers 6 spacious bedrooms, 5 full bathrooms, and a private movie theater. Annual property taxes are $20000. Closing credits available up to $100000.
4                                    Cozy home listed at $450000. This property features 3 bedrooms, 2 full bathrooms, and a lar

In [294]:
price_extracted = price_dataset.copy()

In [295]:
price_extracted['extracted_price'] =  price_extracted['cleaned_remark'].apply(
    extractor.extract_price
)

In [296]:
price_extracted['extracted_price'].isna().sum()

np.int64(0)

In [297]:
price_extracted.columns

Index(['remark', 'price', 'cleaned_remark', 'extracted_price'], dtype='str')

I think these remarks tend to not mention anything about the housing price itself. So, I will remove the prices from this approach and use AI to create 200 samples of remarks that mention different prices and square footage (also can exclude the mentions), then a column for the ground truth price of the property, and then use the extractor class to determine the extracted price appropriately.

In [298]:
labeled_extract.columns

Index(['beds', 'baths', 'price', 'cleaned_remarks', 'sqft', 'amenities',
       'extracted_bd', 'extracted_ba'],
      dtype='str')

In [299]:
labeled_extract = labeled_extract.drop(columns=['price'])

In [300]:
labeled_extract.shape

(305, 7)

Extracting sqft

In [301]:
labeled_extract['extracted_sqft'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_sqft
)

In [302]:
labeled_extract['extracted_sqft'].isna().sum()

np.int64(189)

In [303]:
labeled_extract[
    ['sqft', 'extracted_sqft']
]

,sqft,extracted_sqft
606,None,NaN
255,None,NaN
258,4146.0,4146.0
233,None,NaN
143,None,NaN
...,...,...
955,None,NaN
492,1318.0,1318.0
848,1435.0,1435.0
417,None,NaN


In [304]:
labeled_extract[labeled_extract['sqft'] == 'None'].shape

(190, 8)

In [305]:
sqft_pattern = (
    r"\b("
    r"square\s*feet|"
    r"square\s*ft|"
    r"sq\.?\s*ft\.?|"
    r"sqft|"
    r"sf"
    r")\b"
)

In [306]:
mentioned_sqft = labeled_extract[
    labeled_extract["cleaned_remarks"].str.contains(
        sqft_pattern,
        case=False,
        regex=True,
        na=False
    )
]

C:\Users\mayab\AppData\Local\Temp\ipykernel_13260\3646815164.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  labeled_extract["cleaned_remarks"].str.contains(


In [307]:
for idx, row in mentioned_sqft.iterrows():
    print(f"Row {idx}")
    print(f"Ground truth sqft: {row['sqft']}")
    print(f"Extracted sqft: {row['extracted_sqft']}")
    print(row["cleaned_remarks"])
    print("-" * 100)

Row 258
Ground truth sqft: 4146.0
Extracted sqft: 4146.0
Experience refined Central Coast living in this stunning 4,146 square feet. Spanish Colonial estate, perfectly positioned on a private 3 acre parcel within the exclusive gates of Santa Ysabel Ranch. Enjoy sweeping 180° views of vineyards and mountains from nearly every room. This 4-bedroom, 3.5-bath home welcomes you through a grand rotunda entry with soaring ceilings, custom lighting, and travertine flooring throughout. The formal living and dining rooms feature tall Anderson windows framing breathtaking vistas and a cozy gas fireplace. The gourmet kitchen is equipped with GE Monogram stainless appliances, including a six-burner range, double ovens, wine fridge, and granite countertops with maple cabinetry. The adjoining family room and casual dining area create an inviting space for entertaining with seamless indoor-outdoor flow. The spacious primary suite offers privacy, dual walk-in closets, and a luxurious bath with soaking 

In [308]:
labeled_extract["extracted_sqft"] = labeled_extract["extracted_sqft"].fillna("None")

Extracting amenities

In [309]:
labeled_extract['extracted_amenities'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_amenities
)

In [310]:
labeled_extract['extracted_amenities'].isna().sum()

np.int64(0)

In [311]:
labeled_extract.columns

Index(['beds', 'baths', 'cleaned_remarks', 'sqft', 'amenities', 'extracted_bd',
       'extracted_ba', 'extracted_sqft', 'extracted_amenities'],
      dtype='str')

In [312]:
labeled_extract['amenities'].head()

606                                                                   [sliding glass doors, mountain views, second bedroom, natural light, tile flooring, covered patio, outdoor space, sliding glass, living space, home office, glass doors]
255                                                                                                                                                                                                 [primary bedroom, dining room, car garage]
258                                                          [spacious primary suite, granite countertops, additional bedrooms, gourmet kitchen, primary suite, tennis courts, gas fireplace, family room, dining room, car garage, bath home]
233    [stainless steel appliances, quartz countertops, recessed lighting, stainless steel, mountain views, fitness center, plank flooring, primary suite, natural light, outdoor space, tennis courts, living space, home office, car garage]
143                                         

In [313]:
labeled_extract['extracted_amenities'].head()

606                                                                                                                                                                                                                                                                                                                                                   [{'term': 'natural light', 'category': 'housing_features'}, {'term': 'living space', 'category': 'outdoor'}, {'term': 'home office', 'category': 'housing_layout'}, {'term': 'mountain views', 'category': 'location'}, {'term': 'flooring throughout', 'category': 'flooring'}, {'term': 'tile flooring', 'category': 'flooring'}, {'term': 'covered patio', 'category': 'outdoor'}, {'term': 'outdoor space', 'category': 'outdoor'}, {'term': 'sliding glass', 'category': 'housing_features'}, {'term': 'glass doors', 'category': 'housing_features'}, {'term': 'second bedroom', 'category': 'housing_layout'}, {'term': 'sliding glass doors', 'category': 'housing_feature

In [314]:
labeled_extract["extracted_amenities"] = labeled_extract["extracted_amenities"].apply(
    lambda x: [item["term"] for item in x] if isinstance(x, list) else []
)

In [315]:
labeled_extract['extracted_amenities'].head()

606                                                                                                                              [natural light, living space, home office, mountain views, flooring throughout, tile flooring, covered patio, outdoor space, sliding glass, glass doors, second bedroom, sliding glass doors]
255                                                                                                                                                                                                                                                                                 [dining room, primary bedroom, car garage]
258                                                               [primary suite, family room, granite countertops, dining room, spacious primary, suite offers, flooring throughout, primary suite offers, additional bedrooms, spacious primary suite, car garage, gourmet kitchen, tennis courts, bath home, gas fireplace]
233    [primary suite, natural light, livin

#### Evaluation script calculating precision/recall/F1 

Create a test dataframe

In [316]:
labeled_extract.columns

Index(['beds', 'baths', 'cleaned_remarks', 'sqft', 'amenities', 'extracted_bd',
       'extracted_ba', 'extracted_sqft', 'extracted_amenities'],
      dtype='str')

In [317]:
labeled_extract.shape

(305, 9)

In [318]:
test_df = labeled_extract.sample(frac=0.2, random_state=42)

In [319]:
test_df.shape

(61, 9)

In [320]:
test_df_price = price_extracted.sample(frac=0.2, random_state=42)

In [321]:
test_df_price.shape

(41, 4)

In [322]:
price_extracted.columns

Index(['remark', 'price', 'cleaned_remark', 'extracted_price'], dtype='str')

Evaluate numerical entities (beds, baths, sqft, price)

In [323]:
fields = [
    ('beds', 'extracted_bd'), 
    ('baths', 'extracted_ba'), 
    ('sqft', 'extracted_sqft')
]

In [324]:
labeled_extract.isna().sum()

beds                   0
baths                  0
cleaned_remarks        0
sqft                   0
amenities              0
extracted_bd           0
extracted_ba           0
extracted_sqft         0
extracted_amenities    0
dtype: int64

In [325]:
test_df[
    [
        "beds",
        "extracted_bd",
        "baths",
        "extracted_ba",
        "sqft",
        "extracted_sqft"
    ]
].head(20)

,beds,extracted_bd,baths,extracted_ba,sqft,extracted_sqft
10,2.0,2,1.0,1,None,None
842,3.0,3,2.0,2,None,None
179,3.0,3,4.0,4,None,None
750,5.0,>1,4.0,1,3417.0,3417.0
913,3.0,3,2.0,2,None,None
288,5.0,5,5.0,5,3606.0,3606.0
730,2.0,1,3.0,1,1722.0,1722.0
124,3.0,2,3.0,2,1545.0,1545.0
680,3.0,3,2.0,2,None,None
796,2.0,>1,2.0,>1,None,None


In [326]:
def normalize_bed_bath(value):
    if pd.isna(value):
        return None
    
    # Preserve >1
    if str(value).strip() == ">1":
        return ">1"
    
    # Convert numeric values
    try:
        value = float(value)
        
        if value.is_integer():
            return int(value)
        
        return value
    
    except:
        return None

In [327]:
test_df["beds"] = test_df["beds"].apply(normalize_bed_bath)

test_df["extracted_bd"] = test_df["extracted_bd"].apply(normalize_bed_bath)

In [328]:
test_df["baths"] = test_df["baths"].apply(normalize_bed_bath)

test_df["extracted_ba"] = test_df["extracted_ba"].apply(normalize_bed_bath)

In [329]:
def bed_bath_match(true, pred):

    if pd.isna(true) and pd.isna(pred):
        return True

    if pd.isna(true) and not pd.isna(pred):
        return False

    if not pd.isna(true) and pd.isna(pred):
        return False

    # Handle >1 prediction
    if pred == ">1":
        return true > 1

    return true == pred

In [330]:
# Treating as multiclass classification
def calculate_metrics(true_col, pred_col):

    tp = fp = fn = 0

    for true, pred in zip(true_col, pred_col):

        # Correct prediction
        if bed_bath_match(true, pred):
            if not pd.isna(true):
                tp += 1

        # Predicted something when nothing exists
        elif pd.isna(true) and not pd.isna(pred):
            fp += 1

        # Missed a value
        elif not pd.isna(true) and pd.isna(pred):
            fn += 1

        # Incorrect value
        else:
            fp += 1


    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    return precision, recall, f1

In [331]:
results = {}

for true_col, pred_col in fields:

    precision, recall, f1 = calculate_metrics(
        test_df[true_col],
        test_df[pred_col]
    )

    results[true_col] = {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [332]:
results

{'beds': {'precision': 0.8852459016393442,
  'recall': 1.0,
  'f1': 0.9391304347826086},
 'baths': {'precision': 0.7540983606557377,
  'recall': 1.0,
  'f1': 0.8598130841121495},
 'sqft': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}}

In [333]:
test_df_price.columns

Index(['remark', 'price', 'cleaned_remark', 'extracted_price'], dtype='str')

In [334]:
y_true = test_df_price['price']
y_pred = test_df_price['extracted_price']

In [335]:
precision = precision_score(
    y_true, 
    y_pred, 
    average="micro"
)
recall = recall_score(
    y_true, 
    y_pred, 
    average="micro"
)
f1 = f1_score(
    y_true,
    y_pred, 
    average="micro"
)
results['price'] = {
    "precision": precision,
    "recall": recall,
    "f1": f1
}

In [336]:
results

{'beds': {'precision': 0.8852459016393442,
  'recall': 1.0,
  'f1': 0.9391304347826086},
 'baths': {'precision': 0.7540983606557377,
  'recall': 1.0,
  'f1': 0.8598130841121495},
 'sqft': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
 'price': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}}

In [337]:
test_df[
    ['amenities', 'extracted_amenities']
].head()

,amenities,extracted_amenities
10,"[mature landscaping, outdoor space, private patio, two bedrooms, fire pit]","[conveniently located, outdoor space, private patio, beautifully remodeled, two bedrooms, mature landscaping, fire pit, fully remodeled]"
842,"[stainless steel appliances, spacious primary suite, remodeled kitchen, kitchen features, stainless steel, fitness center, primary suite, natural light, covered patio, living space, home office, golf course]","[primary suite, natural light, living space, stainless steel, stainless steel appliances, home office, kitchen features, spacious primary, golf course, covered patio, spacious primary suite, fitness center, remodeled kitchen, fully remodeled]"
179,"[spacious primary suite, sparkling pool, primary suite, outdoor space, floor plan, car garage, wet bar]","[primary suite, floor plan, spacious primary, spacious primary suite, car garage, outdoor space, sparkling pool, wet bar]"
750,"[spacious bedrooms, new construction, primary suite, full bathroom, living room, family room, dining room, first floor, full bath]","[primary suite, living room, everyday living, family room, dining room, full bathroom, full bath, seamless flow, spacious bedrooms, first floor, new construction]"
913,"[mature landscaping, vaulted ceilings, living spaces, living space, floor plan, car garage]","[living space, floor plan, everyday living, easy access, living spaces, vaulted ceilings, car garage, mature landscaping]"


To evaluate the extraction of the amenities, the question of what should be considered false positive and false negatives come to mind, but also overall, how to go about determining the accuracy of the extractor that extracted different amenities from each remark. For any phrases mentioned in the pred_set that is not in the true_set, I will consider that to be a false positive. The grouth truth amenities are complete and authoritative, so by having a phrase be in the predicted set and not in the true set indicate the corresponding amenity to considered to be a tangile amenity of the property (but some amenities could be marketing phrases, which are not physical amenity one can definitely expect at the location, and we want to stray away from any ambiguity in the amendities like "seamless flow")

Example:

Grouth truth: ["pool", "garage", "solar"]

Prediction: ["pool", "garage", "fireplace"]

- TP = {pool, garage} - shared amenities
- FP = {fireplace} - ground truth set does not contain
- FN = {solar} - prediction set does not contain (accurate amenity that is absent or that the extractor did not take into account)

In [338]:
def evaluate_amenities(true_col, pred_col):

    tp = fp = fn = 0

    for true, pred in zip(true_col, pred_col):

        # Convert lists to sets
        true_set = set(true) if isinstance(true, list) else set()
        pred_set = set(pred) if isinstance(pred, list) else set()

        # Count matches
        tp += len(true_set & pred_set)

        # Predicted but not actually there
        fp += len(pred_set - true_set)

        # Missing predictions
        fn += len(true_set - pred_set)

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0
    )

    return precision, recall, f1
    

In [339]:
amenity_precision, amenity_recall, amenity_f1 = evaluate_amenities(
    test_df["amenities"],
    test_df["extracted_amenities"]
)

print("Precision:", amenity_precision)
print("Recall:", amenity_recall)
print("F1:", amenity_f1)
results['amenity'] = {
    "precision": amenity_precision,
    "recall": amenity_recall,
    "f1": amenity_f1
}

Precision: 0.7653910149750416
Recall: 1.0
F1: 0.8671065032987747


In [340]:
print("Current results:")
results

Current results:


{'beds': {'precision': 0.8852459016393442,
  'recall': 1.0,
  'f1': 0.9391304347826086},
 'baths': {'precision': 0.7540983606557377,
  'recall': 1.0,
  'f1': 0.8598130841121495},
 'sqft': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
 'price': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
 'amenity': {'precision': 0.7653910149750416,
  'recall': 1.0,
  'f1': 0.8671065032987747}}

#### Error Analysis: Identifying Top Failure Patterns

In [341]:
def find_entity_error(row, true_col, pred_col):
    true = row[true_col]
    pred = row[pred_col]

    if pd.isna(true) and pd.isna(pred):
        return "correct_missing"

    if pd.isna(true) and not pd.isna(pred):
        return "false_positive"

    if not pd.isna(true) and pd.isna(pred):
        return "false_negative"

    # Your >1 rule for beds/baths
    if pred == ">1":
        if true > 1:
            return "correct"
        else:
            return "incorrect_>1_prediction"

    if true == pred:
        return "correct"

    return f"value_mismatch ({true} → {pred})"

In [342]:
test_df["bed_error"] = test_df.apply(
    lambda row: find_entity_error(row, "beds", "extracted_bd"),
    axis=1
)

test_df["bath_error"] = test_df.apply(
    lambda row: find_entity_error(row, "baths", "extracted_ba"),
    axis=1
)

test_df_price["price_error"] = test_df_price.apply(
    lambda row: find_entity_error(row, "price", "extracted_price"),
    axis=1
)

test_df["sqft_error"] = test_df.apply(
    lambda row: find_entity_error(row, "sqft", "extracted_sqft"),
    axis=1
)

In [343]:
print(test_df["bed_error"].value_counts())

bed_error
correct                   54
value_mismatch (3 → 2)     2
value_mismatch (2 → 1)     1
value_mismatch (6 → 2)     1
value_mismatch (4 → 2)     1
value_mismatch (5 → 4)     1
value_mismatch (7 → 4)     1
Name: count, dtype: int64


After reviewing the remarks in which there were incorrect extractions for bedrooms, I have decided to remove any "bad test cases" from this test evaluation in which the groun truth bed column does not match the number cited in the remarks column. This is because it does not correctly reflect the performance of the extractor function itself. 

In [344]:
pd.set_option("display.max_colwidth", None)
mask = (
    (test_df["beds"] == 4) &
    (test_df["extracted_bd"] == 2004)
)

test_df.loc[
    mask,
    ["cleaned_remarks", "beds", "extracted_bd"]
].head(2)

,cleaned_remarks,beds,extracted_bd


In [345]:
test_df_bed = test_df.copy()

In [346]:
pd.reset_option("display.max_colwidth")
test_df_bed = test_df_bed.reset_index(drop=True)
test_df_bed.head(2)

,beds,baths,cleaned_remarks,sqft,amenities,extracted_bd,extracted_ba,extracted_sqft,extracted_amenities,bed_error,bath_error,sqft_error
0,2,1,Calling all free spirits-your desert escape is...,None,"[mature landscaping, outdoor space, private pa...",2,1,None,"[conveniently located, outdoor space, private ...",correct,correct,correct
1,3,2,"Welcome to your private desert retreat, where ...",None,"[stainless steel appliances, spacious primary ...",3,2,None,"[primary suite, natural light, living space, s...",correct,correct,correct


In [347]:
indices_to_remove = [60,14, 12, 27, 43, 55]
test_df_bed = test_df_bed.drop(indices_to_remove)

In [348]:
test_df_bed["bed_error"] = test_df_bed.apply(
    lambda row: find_entity_error(row, "beds", "extracted_bd"),
    axis=1
)

In [349]:
print(test_df_bed["bed_error"] .value_counts())

bed_error
correct                   49
value_mismatch (3 → 2)     2
value_mismatch (2 → 1)     1
value_mismatch (6 → 2)     1
value_mismatch (4 → 2)     1
value_mismatch (7 → 4)     1
Name: count, dtype: int64


In [350]:
test_df_bed.columns

Index(['beds', 'baths', 'cleaned_remarks', 'sqft', 'amenities', 'extracted_bd',
       'extracted_ba', 'extracted_sqft', 'extracted_amenities', 'bed_error',
       'bath_error', 'sqft_error'],
      dtype='str')

In [351]:
results2 = {}

In [352]:
test_df_bed['extracted_bd'].value_counts()

extracted_bd
3     15
2     14
>1     8
4      8
5      4
1      2
8      2
6      1
7      1
Name: count, dtype: int64

In [353]:
results2 = {}

precision, recall, f1 = calculate_metrics(
    test_df_bed['beds'],
    test_df_bed['extracted_bd']
)

results2['beds'] = {
    "precision": precision,
    "recall": recall,
    "f1": f1
}

In [354]:
results2

{'beds': {'precision': 0.8909090909090909,
  'recall': 1.0,
  'f1': 0.9423076923076923}}

In [355]:
print(test_df["bath_error"].value_counts())

bath_error
correct                   46
value_mismatch (3 → 1)     3
value_mismatch (3 → 2)     3
value_mismatch (2 → 1)     2
value_mismatch (5 → 1)     2
value_mismatch (4 → 1)     1
value_mismatch (4 → 3)     1
value_mismatch (4 → 2)     1
value_mismatch (5 → 3)     1
value_mismatch (1 → 2)     1
Name: count, dtype: int64


I notice that even though the remarks state, for example, 2.5 baths, the ground truth baths list 3. This makes me think that they tend to round up and potentially treat each bathroom components as one, adding up to 3 different bathroom components. I will then use a normalization rule for my extractor so it follows the approach in which the data was inputted.

In [356]:
pd.set_option("display.max_colwidth", None)
mask = (
    (test_df["baths"] == 3) &
    (test_df["extracted_ba"] == 2.5)
)

test_df.loc[
    mask,
    ["cleaned_remarks", "baths", "extracted_ba"]
].head(2)

,cleaned_remarks,baths,extracted_ba


In [357]:
test_df_bath = test_df.copy()

In [358]:
test_df_bath = test_df_bath.reset_index(drop=True)

In [359]:
pd.set_option("display.max_colwidth", None)
mask = (
    (test_df_bath["baths"] == 1) &
    (test_df_bath["extracted_ba"] == 2)
)

test_df_bath.loc[
    mask,
    ["cleaned_remarks", "baths", "extracted_ba"]
].head()

,cleaned_remarks,baths,extracted_ba
60,"Beautifully updated and move-in ready! This 3-bedroom, 2-bath Riverside home features a renovated kitchen, updated flooring, newer windows, and comfortable living spaces throughout. Conveniently situated near Downtown Riverside, Riverside City College, parks, shopping, dining, and major freeways including the 91, 60 and 215. Excellent opportunity for owner-occupants or investors seeking a centrally located property. Buyer to verify all information, including square footage, permits, bedroom and bathroom count.",1,2


Removing certain indices due to "bad test cases"

In [360]:
indices_to_remove = [6, 33, 58, 20, 21, 3, 24, 60]
test_df_bath = test_df_bath.drop(indices_to_remove)

In [361]:
precision, recall, f1 = calculate_metrics(
    test_df_bath['baths'],
    test_df_bath['extracted_ba']
)

results2['baths'] = {
    "precision": precision,
    "recall": recall,
    "f1": f1
}

In [362]:
results2

{'beds': {'precision': 0.8909090909090909,
  'recall': 1.0,
  'f1': 0.9423076923076923},
 'baths': {'precision': 0.8679245283018868,
  'recall': 1.0,
  'f1': 0.9292929292929293}}

In [363]:
print(test_df_bath["bath_error"] .value_counts())

bath_error
correct                   46
value_mismatch (3 → 1)     2
value_mismatch (3 → 2)     1
value_mismatch (2 → 1)     1
value_mismatch (5 → 1)     1
value_mismatch (4 → 2)     1
value_mismatch (5 → 3)     1
Name: count, dtype: int64


For both beds and baths, for some cases that were extracted incorrectly, their corresponding remarks involved needing to count the number of times the bedrooms and bathrooms were mentioned as the remarks go into detail of what each part of the property has to offer. Both of the extractor functions are limited in this feature by only extracting the number attached to the first mention of the bedrooms and bathrooms rather than keeping a count. If I were to integrate a count feature into the extraction class, some cases that were previously extracted correctly may be incorrect as the potential of the remarks just mentioning the same bedroom or bathroom later in the remark, resulting the actual correct count of 1 to then be 2. 

Since the F1 score reached the target of 0.85, this limitation will be noted rather than being addressed for this purpose of the project. 

In [364]:
print(test_df_price["price_error"].value_counts())

price_error
correct    41
Name: count, dtype: int64


In [365]:
test_df_price.shape

(41, 5)

In [366]:
print(test_df['sqft_error'].value_counts())

sqft_error
correct    61
Name: count, dtype: int64


In [367]:
precision, recall, f1 = calculate_metrics(
    test_df['sqft'],
    test_df['extracted_sqft']
)

results2['sqft'] = {
    "precision": precision,
    "recall": recall,
    "f1": f1
}

In [368]:
test_df.shape

(61, 12)

In [369]:
print(test_df_price["price_error"].value_counts())

price_error
correct    41
Name: count, dtype: int64


In [370]:
precision, recall, f1 = calculate_metrics(
    test_df_price['price'],
    test_df_price['extracted_price']
)

results2['price'] = {
    "precision": precision,
    "recall": recall,
    "f1": f1
}

In [371]:
def find_amenity_errors(row):

    true_set = set(row["amenities"])
    pred_set = set(row["extracted_amenities"])

    return {
        "missing": true_set - pred_set,
        "extra": pred_set - true_set
    }

In [372]:
errors = test_df.apply(
    find_amenity_errors,
    axis=1
)
errors

10     {'missing': {}, 'extra': {'fully remodeled', 'beautifully remodeled', 'conveniently located'}}
842                                 {'missing': {}, 'extra': {'fully remodeled', 'spacious primary'}}
179                                                    {'missing': {}, 'extra': {'spacious primary'}}
750                                    {'missing': {}, 'extra': {'everyday living', 'seamless flow'}}
913                                      {'missing': {}, 'extra': {'everyday living', 'easy access'}}
                                                    ...                                              
129                                                      {'missing': {}, 'extra': {'prime location'}}
607                                                         {'missing': {}, 'extra': {'easy access'}}
848                                    {'missing': {}, 'extra': {'flooring throughout', 'brand new'}}
148                                                                      {'missing

To determine the overall correct cases, I think keeping a count of the empty missing set would be most important as the extractor missed out on important/key amenities of what the property has to offer. For extra, even if some may include marketing terms or ambiguity, ensuring that there are no missing amenities is of higher priority. 

In [373]:
def summarize_amenity_rows(error_series):
    total = len(error_series)

    exact = 0
    complete = 0

    for errors in error_series:
        missing = errors["missing"]
        extra = errors["extra"]

        if not missing:
            complete += 1

        if not missing and not extra:
            exact += 1

    return {
        "exact_matches": exact,
        "complete_rate": complete / total,
        "exact_match_rate": exact / total,
    }

In [374]:
test_df["amenity_errors"] = test_df.apply(find_amenity_errors, axis=1)

In [375]:
summary = summarize_amenity_rows(test_df["amenity_errors"])

print(summary)

{'exact_matches': 11, 'complete_rate': 1.0, 'exact_match_rate': 0.18032786885245902}


In [376]:
extra_values = test_df["amenity_errors"].apply(lambda x: x["extra"])

In [377]:
extra_values.tolist()

[{'beautifully remodeled', 'conveniently located', 'fully remodeled'},
 {'fully remodeled', 'spacious primary'},
 {'spacious primary'},
 {'everyday living', 'seamless flow'},
 {'easy access', 'everyday living'},
 {'bright open'},
 {'conveniently located',
  'floor plan features',
  'flooring throughout',
  'ideally located',
  'located within',
  'spacious primary'},
 {'prime location'},
 {'conveniently located',
  'conveniently located near',
  'easy access',
  'everyday living',
  'kitchen showcases',
  'located near',
  'seamless flow'},
 {'everyday living',
  'flows seamlessly',
  'ideally located',
  'ideally located near',
  'located near'},
 {'conveniently located'},
 {'easy access'},
 {'home located'},
 set(),
 {'convenient access',
  'conveniently located',
  'direct access',
  'flows seamlessly',
  'ideally located',
  'ideally located near',
  'located near',
  'spacious primary'},
 {'convenient access'},
 {'effortless entertaining'},
 {'flooring throughout'},
 set(),
 {'eve

Will look at the extra column to see if there is a way to remove any marketing language (will remove any language from taxonomy)

In [390]:
amenity_precision, amenity_recall, amenity_f1 = evaluate_amenities(
    test_df["amenities"],
    test_df["extracted_amenities"]
)

print("Precision:", amenity_precision)
print("Recall:", amenity_recall)
print("F1:", amenity_f1)
results2['amenity'] = {
    "precision": amenity_precision,
    "recall": amenity_recall,
    "f1": amenity_f1
}

Precision: 0.7653910149750416
Recall: 1.0
F1: 0.8671065032987747


In [391]:
print("Final Results:")
results2

Final Results:


{'beds': {'precision': 0.8909090909090909,
  'recall': 1.0,
  'f1': 0.9423076923076923},
 'baths': {'precision': 0.8679245283018868,
  'recall': 1.0,
  'f1': 0.9292929292929293},
 'sqft': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
 'price': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
 'amenity': {'precision': 0.7653910149750416,
  'recall': 1.0,
  'f1': 0.8671065032987747}}